In [1]:
# import subprocess

# result = subprocess.run(
#     [
#         "java",
#         "-cp",
#         "publicationclassification.jar",
#         "nl.cwts.publicationclassification.run.PublicationClassificationCreator",
#     ],
#     capture_output=True,
#     text=True,
# )
# print(result.stdout)
# print(result.stderr)

In [2]:
import subprocess, sys, os

'''
This output is:
 1. pubs.txt - Paper list for CWTS tool
    Columns: int_id  paper identifier for cwts, core_pub (always 1 as not using core feature, but it has to be in)

 2. cit_links.txt - Citation network edges
    Columns: 
    int_id1 - citing paper, 
    int_id2 - cited paper, 
    weight - citation strength 0-2 higher= stronger
    Note: Each edge appears twice (A→B and B→A) for undirected format
        paper 5 cites Paper 12  →  row: 5, 12, 0.85
        Paper 12 cites Paper 5  →  row: 12, 5, 0.85  (same edge, reversed)

 3. pub_metadata.txt - Paper details lookup table
    Columns: int_id, pub_id, is_frontiers, journal, date, title
    - int_id: sequential CWTS ID (joins to classification.txt)
    - pub_id: airak PublicationId (joins to BigQuery tables)
 JOIN KEY: int_id links all files together, this is cwts identifier
'''
print('ere')
env = os.environ.copy()
print('ere')
env["START_YEAR"] = "2020"
env["END_YEAR"] = "2026"
env["NETWORK_MODE"] = "global"
print('ere')
result = subprocess.run(
    [sys.executable, "cwts_export.py"], capture_output=True, text=True, env=env
)
print('ere')
print(result.stdout)
print(result.stderr)
print('last')

ere
ere
ere


KeyboardInterrupt: 

In [ ]:
import subprocess
import datetime
import os

# --- Parameters ---
params = {
    "largest_component_only": "true",
    "iterations": "1000",
    "micro_resolution": "2e-4",
    "micro_min_cluster_size": "10000",
    "meso_resolution": "4.9e-7",
    "meso_min_cluster_size": "10000",
    "macro_resolution": "2.2e-8",
    "macro_min_cluster_size": "200000",
}

input_files = {
    "pubs": "cwts_output/pubs.txt",
    "cit_links": "cwts_output/cit_links.txt",
    "output": "cwts_output/classification.txt",
    "jar": "publicationclassification.jar",
}

# --- Run ---
run_timestamp = datetime.datetime.now()

result = subprocess.run(
    [
        "java", "-cp", input_files["jar"],
        "nl.cwts.publicationclassification.run.PublicationClassificationCreator",
        input_files["pubs"],
        input_files["cit_links"],
        input_files["output"],
        params["largest_component_only"],
        params["iterations"],
        params["micro_resolution"],
        params["micro_min_cluster_size"],
        params["meso_resolution"],
        params["meso_min_cluster_size"],
        params["macro_resolution"],
        params["macro_min_cluster_size"],
    ],
    capture_output=True,
    text=True,
)

# --- Log ---
os.makedirs("logs", exist_ok=True)
log_path = f"logs/cwts_run_{run_timestamp.strftime('%Y%m%d_%H%M%S')}.log"

with open(log_path, "w") as f:
    f.write(f"CWTS Publication Classification Run\n")
    f.write(f"{'='*50}\n")
    f.write(f"Timestamp : {run_timestamp.isoformat()}\n\n")

    f.write(f"Input Files\n{'-'*30}\n")
    for k, v in input_files.items():
        f.write(f"  {k:<20}: {v}\n")

    f.write(f"\nParameters\n{'-'*30}\n")
    for k, v in params.items():
        f.write(f"  {k:<26}: {v}\n")

    f.write(f"\nReturn Code: {result.returncode}\n")

    f.write(f"\nSTDOUT\n{'-'*30}\n")
    f.write(result.stdout or "(empty)\n")

    f.write(f"\nSTDERR\n{'-'*30}\n")
    f.write(result.stderr or "(empty)\n")

print(f"Log written to: {log_path}")
print(result.stdout)
if result.stderr:
    print(result.stderr)

In [1]:
import pandas as pd
df = pd.read_csv(
    "cwts_output/classification.txt",
    sep="\t",
    header=None,
    names=["pub_no", "micro", "meso", "macro"],
)

print(f"Total classified: {len(df):,}")
for level in ["micro", "meso", "macro"]:
    vc = df[level].value_counts()
    print(f"\n{level.upper()}: {len(vc):,} clusters")
    print(f"  Largest : {vc.iloc[0]:,} ({vc.iloc[0]/len(df)*100:.1f}%)")
    print(f"  Smallest: {vc.iloc[-1]:,}")
    print(f"  Median  : {vc.median():.0f}")

Total classified: 30,692,509

MICRO: 6,840 clusters
  Largest : 45,813 (0.1%)
  Smallest: 2,000
  Median  : 3670

MESO: 1,616 clusters
  Largest : 72,694 (0.2%)
  Smallest: 10,001
  Median  : 16910

MACRO: 61 clusters
  Largest : 3,586,875 (11.7%)
  Smallest: 21,690
  Median  : 168704


### Labelling with GPT

In [ ]:
import label_clusters

# Run the script
label_clusters.main()

In [ ]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("cwts_output")
classif_path = OUTPUT_DIR / "classification.txt"
titles_path = OUTPUT_DIR / "pub_titles.txt"
classif = pd.read_csv(
    classif_path,
    sep="\t",
    header=None,
    names=["pub_no", "micro", "meso", "macro"],
)

print("Loading titles...")
titles = pd.read_csv(
    titles_path,
    sep="\t",
    header=None,
    names=["pub_no", "title"],
)

merged = classif.merge(titles, on="pub_no")
print(f"Merged: {len(merged):,} publications")

# merged.sample(5000).to_csv('full_merged.csv')

In [ ]:
merged.sample(5000).to_csv("merged_sample.csv")

In [ ]:
pd.set_option("display.max_colwidth", None)
merged[['macro','title']].sample(500)

### looking at scope

In [ ]:
import pandas as pd
from pathlib import Path

core = pd.read_csv(
    "cwts_output/frontiers_core.txt",
    sep="\t",
    header=None,
    names=["pub_no", "is_frontiers", "journal"],
)

print(core["journal"].unique())

In [ ]:
import importlib
import journal_scope

# Override config variables
journal_scope.SCOPE_LEVEL = "macro"
journal_scope.SCOPE_THRESHOLD = 0.80
journal_scope.MIN_PAPERS = 50
journal_scope.USE_GPT = False
journal_scope.OUTPUT_DIR = Path("cwts_output")

journal_scope.TARGET_JOURNALS = [
    "Frontiers in Immunology",
    "Frontiers in Public Health",
    "Frontiers in Medicine",
    "Frontiers in Oncology",
    "Frontiers in Psychology",
]

# Run
journal_scope.main()

In [ ]:
pd.set_option("display.max_colwidth", 50)
pd.read_csv('cwts_output/journal_scope.csv')